# Reproducing GIFT-Eval results for Yug

This notebook forecasts GIFT-Eval benchmark datasets with Yug and scores them with the official GIFT-Eval harness. This notebook lives in `yug` itself: the model and predictor come entirely from this repo's own `yug` package (nothing copied in, nothing to keep in sync) — the only external pieces are the GIFT-Eval benchmark harness (`gift_eval`, an ordinary pip dependency, same as installing `gluonts` or `torch`) and the benchmark data itself, which is too large to ship in this repo and has to be downloaded separately.

Before running this notebook:

1. Download the GIFT-Eval benchmark data and set `GIFT_EVAL` to point at it — see [the gift-eval README](https://github.com/SalesforceAIResearch/gift-eval#installation). `notebooks/.env` (gitignored) is the usual place for `GIFT_EVAL=/path/to/GiftEval`.
2. `notebooks/dataset_properties.json` (bundled next to this notebook) supplies each dataset's domain and native dimensionality for the results table — it is static benchmark metadata, not code.

For brevity this notebook evaluates two datasets end to end (one short-horizon-only, one that also has medium/long terms and is natively multivariate); change `SHORT_DATASETS` / `MED_LONG_DATASETS` below to run the full 97-configuration benchmark — the code path is identical either way.

**Install**

```bash
pip install "yug[gluonts]"
pip install "salesforce-gift-eval @ git+https://github.com/SalesforceAIResearch/gift-eval.git"
pip install python-dotenv
```

The `gift-eval` install fetches straight from GitHub (it isn't on PyPI) but leaves nothing checked out on disk — pip builds and discards its own clone, same as any other VCS dependency.

Weights load from the Hugging Face Hub (`birlaailabs/yug`) via `YugPredictor.from_pretrained`, or from a local checkpoint directory containing `config.json` + `model.safetensors` — useful before the Hub repo is public, or for evaluating an unpublished checkpoint.

In [ ]:
! pip install "yug[gluonts]"
! pip install "salesforce-gift-eval @ git+https://github.com/SalesforceAIResearch/gift-eval.git"
! pip install python-dotenv

In [ ]:
import json

from dotenv import load_dotenv

# Load environment variables (GIFT_EVAL path) from notebooks/.env
load_dotenv()

# SHORT_DATASETS = "m4_yearly m4_quarterly m4_monthly m4_weekly m4_daily m4_hourly electricity/15T electricity/H electricity/D electricity/W solar/10T solar/H solar/D solar/W hospital covid_deaths us_births/D us_births/M us_births/W saugeenday/D saugeenday/M saugeenday/W temperature_rain_with_missing kdd_cup_2018_with_missing/H kdd_cup_2018_with_missing/D car_parts_with_missing restaurant hierarchical_sales/D hierarchical_sales/W LOOP_SEATTLE/5T LOOP_SEATTLE/H LOOP_SEATTLE/D SZ_TAXI/15T SZ_TAXI/H M_DENSE/H M_DENSE/D ett1/15T ett1/H ett1/D ett1/W ett2/15T ett2/H ett2/D ett2/W jena_weather/10T jena_weather/H jena_weather/D bitbrains_fast_storage/5T bitbrains_fast_storage/H bitbrains_rnd/5T bitbrains_rnd/H bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H"
SHORT_DATASETS = "m4_weekly"

# MED_LONG_DATASETS = "electricity/15T electricity/H solar/10T solar/H kdd_cup_2018_with_missing/H LOOP_SEATTLE/5T LOOP_SEATTLE/H SZ_TAXI/15T M_DENSE/H ett1/15T ett1/H ett2/15T ett2/H jena_weather/10T jena_weather/H bitbrains_fast_storage/5T bitbrains_rnd/5T bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H"
MED_LONG_DATASETS = "bizitobs_l2c/H"

# Union of short and med_long datasets
all_datasets = list(set(SHORT_DATASETS.split() + MED_LONG_DATASETS.split()))

dataset_properties_map = json.load(open("dataset_properties.json"))

from gluonts.ev.metrics import (
    MAE,
    MAPE,
    MASE,
    MSE,
    MSIS,
    ND,
    NRMSE,
    RMSE,
    SMAPE,
    MeanWeightedSumQuantileLoss,
)

# Same 11 metrics and naming convention as the gift-eval reference notebooks,
# so results are directly comparable on the leaderboard.
metrics = [
    MSE(forecast_type="mean"),
    MSE(forecast_type=0.5),
    MAE(),
    MASE(),
    MAPE(),
    SMAPE(),
    MSIS(),
    RMSE(),
    NRMSE(),
    ND(),
    MeanWeightedSumQuantileLoss(
        quantile_levels=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
    ),
]

## Predictor

The predictor is [`yug.gluonts_adapter.YugPredictor`](../src/yug/gluonts_adapter.py) — part of this repo's own package, built for exactly this purpose. It wraps `YugPipeline` (load once, `predict()` many times) and converts each forecast into a `gluonts.model.forecast.QuantileForecast`, which is exactly what `evaluate_forecasts` below expects. We import it rather than redefining it here so the benchmark always exercises the same code path a user gets from `pip install yug` — no drift between what's evaluated and what's shipped.

Yug forecasts a fixed set of quantile levels (`0.1 ... 0.9`, matching the metric above) rather than a continuous distribution; horizons longer than one forward pass are produced by `YugPipeline.predict`'s autoregressive rollout, which samples a trajectory from the predicted band and feeds the *sample* (not the median) back in as context — see the pipeline's own docstring for why that matters for the width of the returned interval at long horizons.

In [ ]:
import logging

from yug.gluonts_adapter import YugPredictor

logger = logging.getLogger("Yug Predictor")
logger.setLevel(logging.INFO)

# Hugging Face Hub id once published; until then point this at a local
# checkpoint directory containing config.json + model.safetensors, e.g.
# MODEL_ID = "./.local_checkpoint"
MODEL_ID = "birlaailabs/yug"

DEVICE = "cuda"  # "cpu" also works, just much slower
NUM_SAMPLES = 100  # rollout paths; matches YugPipeline's PredictionConfig default
ENGINE = "cached"  # "cached" (fast, ~1e-6 drift) or "exact" (bitwise-identical, slow)
SEED = 0
# History fed to the model: the latest 2048 points of every series (shorter
# series are used whole). Yug was trained on 2048-point windows and its
# published scores were produced at 2048. That is also YugPredictor's default;
# it is set explicitly so the benchmark cannot drift if the default changes.
CONTEXT_LENGTH = 2048

## Evaluation

`evaluate_on_dataset` below loads each `(dataset, term)` with `gift_eval.data.Dataset`, builds a `YugPredictor`, forecasts every test window, and scores the result with gluonts' `evaluate_forecasts`. The first column of the output table is the dataset config name, following the gift-eval convention:

```python
f"{dataset_name}/{freq}/{term}"
```

**Univariate only.** Several GIFT-Eval datasets (`bizitobs_l2c`, `LOOP_SEATTLE`, `ett1`/`ett2`, ...) are natively multivariate. `YugPipeline` does support conditioning a forecast on covariate channels (`predict(..., context=multivariate_array)`), but that rollout only forecasts the *target* row — it does not itself explode a multivariate series into one forecast per channel the way `evaluate_forecasts` expects for a multivariate `target_dim`. So this notebook loads every genuinely multivariate dataset (`target_dim > 1`) with `to_univariate=True`: each channel is scored as its own independent series. (A native univariate dataset must stay `to_univariate=False` — gift-eval's `MultivariateToUnivariate` transform assumes a >1-D target and breaks on a 1-D one.) That matches how most foundation-model submissions on the leaderboard are evaluated, and keeps the numbers unambiguous; `num_variates` in the output table still reports the dataset's native dimensionality — only the modelling choice is univariate.

In [ ]:
all_datasets

In [ ]:
import logging


class WarningFilter(logging.Filter):
    def __init__(self, text_to_filter):
        super().__init__()
        self.text_to_filter = text_to_filter

    def filter(self, record):
        return self.text_to_filter not in record.getMessage()


gts_logger = logging.getLogger("gluonts.model.forecast")
gts_logger.addFilter(
    WarningFilter("The mean prediction is not stored in the forecast data")
)

In [ ]:
from pathlib import Path

import pandas as pd
from gluonts.model import evaluate_forecasts
from gluonts.time_feature import get_seasonality

from gift_eval.data import Dataset

# Gitignored (results/ at any depth) -- this is a local reproduction run, not
# a leaderboard submission artifact.
output_dir = Path("results/Yug")
output_dir.mkdir(parents=True, exist_ok=True)

pretty_names = {
    "saugeenday": "saugeen",
    "temperature_rain_with_missing": "temperature_rain",
    "kdd_cup_2018_with_missing": "kdd_cup_2018",
    "car_parts_with_missing": "car_parts",
}
pretty_model_name = {
    MODEL_ID: "Yug",
}


def evaluate_on_dataset(ds_name: str, ds_term: str):
    # MultivariateToUnivariate errors on an already-univariate series (its
    # per-channel split assumes a >1-D target), so only apply it when the
    # source really is multivariate -- same guard chronos-2.ipynb uses.
    is_multivariate_source = (
        Dataset(name=ds_name, term=ds_term, to_univariate=False).target_dim > 1
    )
    dataset = Dataset(name=ds_name, term=ds_term, to_univariate=is_multivariate_source)

    # NOTE: do not call len(dataset.test_data) here -- gift-eval backs
    # test_data.input with a single-pass generator once to_univariate
    # explodes multivariate channels, so counting it first silently
    # exhausts it and every forecast below comes back empty/corrupt.

    predictor = YugPredictor.from_pretrained(
        MODEL_ID,
        device_map=DEVICE,
        freq=dataset.freq,
        prediction_length=dataset.prediction_length,
        num_samples=NUM_SAMPLES,
        context_length=CONTEXT_LENGTH,
        engine=ENGINE,
        seed=SEED,
    )

    forecasts = list(
        predictor.predict(dataset.test_data.input, dataset.prediction_length)
    )
    season_length = get_seasonality(dataset.freq)
    return (
        evaluate_forecasts(
            forecasts,
            test_data=dataset.test_data,
            metrics=metrics,
            batch_size=1024,
            axis=None,
            mask_invalid_label=True,
            allow_nan_forecast=False,
            seasonality=season_length,
        )
        .reset_index(drop=True)
        .to_dict(orient="records")
    )


all_results = []
for ds_num, ds_name in enumerate(all_datasets):
    logger.info(f"Processing dataset: {ds_name} ({ds_num + 1} of {len(all_datasets)})")
    terms = ["short", "medium", "long"]
    for term in terms:
        if term != "short" and ds_name not in MED_LONG_DATASETS.split():
            continue

        if "/" in ds_name:
            ds_key, ds_freq = ds_name.split("/")
            ds_key = pretty_names.get(ds_key.lower(), ds_key.lower())
        else:
            ds_key = pretty_names.get(ds_name.lower(), ds_name.lower())
            ds_freq = dataset_properties_map[ds_key]["frequency"]
        ds_config = f"{ds_key}/{ds_freq}/{term}"

        logger.info(f"Generating forecasts for {ds_config}")
        all_results.append(
            (
                evaluate_on_dataset(ds_name, term),
                ds_config,
                dataset_properties_map[ds_key]["domain"],
                dataset_properties_map[ds_key]["num_variates"],
            )
        )


result_df_rows = []
for result_metrics, ds_config, domain, num_variates in all_results:
    result_metrics = {f"eval_metrics/{k}": v for k, v in result_metrics[0].items()}
    result_df_rows.append(
        {
            "dataset": ds_config,
            "model": pretty_model_name.get(MODEL_ID, MODEL_ID),
            **result_metrics,
            "domain": domain,
            "num_variates": num_variates,
        }
    )

results_df = pd.DataFrame(result_df_rows).sort_values(by="dataset")
results_df.to_csv(output_dir / "all_results.csv", index=False)
logger.info(f"Results have been written to {output_dir / 'all_results.csv'}.")
results_df